In [ ]:
# hide
import numpy as np
import pyquist as pq

In [ ]:
def ping_pong_delay(x: np.ndarray, M: int, g: float) -> np.ndarray:
    """A stereo feedback delay whose echoes bounce between the left and right channels."""
    left_buffer, right_buffer = np.zeros(M), np.zeros(M)   # two delay lines
    i = 0
    y = np.zeros((len(x), 2))
    for n in range(len(x)):
        y[n, 0] = x[n] + g * right_buffer[i]   # echoes from the right return on the left
        y[n, 1] = g * left_buffer[i]           # echoes from the left return on the right
        left_buffer[i], right_buffer[i] = y[n, 0], y[n, 1]
        i = (i + 1) % M
    return y


ball = pq.Audio.from_file("../assets/ping-pong.wav")
f_s = ball.sample_rate
hit = ball.segment(offset=0.5, duration=0.3).samples[:, 0]   # a single bounce
x = np.concatenate([hit, np.zeros(3 * f_s)])                 # room for the echoes
pq.play(pq.Audio(ping_pong_delay(x, M=round(0.3 * f_s), g=0.7), f_s), normalize=True)